# ⚡ Project Aarambh (आरम्भ) Multi-Modal Streaming Gateway (Production)
### Google Colab GPU (Tesla T4) <==> Local Streamlit Studio Bridge
---
**Hardware & Architecture Rules**:
- **Hardware**: NVIDIA Tesla T4 16GB GPU (CUDA 12)
- **Persistent Storage**: `/content/drive/MyDrive/ComfyUI` caching (Zero re-download)
- **Anti-Disconnect**: Background WebAudio oscillator keeps connection alive for up to 12 hours
- **Models Supported**: DreamShaper 8 (Image), LTX-Video 2.5 (Video), Hunyuan3D-2 (3D Mesh)
- **Resilient Tunnel**: Ngrok (Primary) + Cloudflared (Auto-Fallback) with automatic zombie cleanup
- **Streaming Transport**: WebSocket binary streaming with `out[8:]` frame slicing

> **ONE-CLICK RUN**: Click **Runtime -> Run all** (Ctrl+F9). Copy the generated **PUBLIC GATEWAY URL** from Cell 5 into your Local Studio!

In [ ]:
# @title 1. Verify NVIDIA T4 GPU & CUDA Accelerator
import torch, sys, subprocess

if not torch.cuda.is_available():
    print('\n' + '!'*70)
    print('  ❌ FATAL: NVIDIA GPU IS NOT ACTIVE!')
    print('  👉 Switch runtime: Runtime -> Change runtime type -> T4 GPU -> Save')
    print('!'*70 + '\n')
    raise SystemExit('HALTED: T4 GPU accelerator required.')

gpu_name = torch.cuda.get_device_name(0)
gpu_vram = round(torch.cuda.get_device_properties(0).total_memory / 1048576)
print(f'[✓] ACTIVE ACCELERATOR: {gpu_name} ({gpu_vram} MB VRAM)')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
# @title 2. Persistent Google Drive Mount & Silent Anti-Disconnect WebAudio
import os, sys
from IPython.display import HTML, display
import time

# 1. Silent HTML5 WebAudio Keep-Alive (Prevents Colab from sleeping)
display(HTML('''
<script>
function keepAliveAudio() {
    try {
        let ctx = new (window.AudioContext || window.webkitAudioContext)();
        let osc = ctx.createOscillator();
        let gain = ctx.createGain();
        gain.gain.value = 0.0001; // Silent tone
        osc.connect(gain);
        gain.connect(ctx.destination);
        osc.start();
        console.log('⚡ Project Aarambh Anti-Disconnect WebAudio Active');
    } catch(e) { console.warn(e); }
}
keepAliveAudio();
setInterval(keepAliveAudio, 180000);
</script>
'''))
print('[✓] Silent Anti-Disconnect WebAudio Heartbeat active.')

# 2. Persistent Google Drive Mount
DRIVE_MOUNTED = False
try:
    if os.path.exists('/content/drive/MyDrive'):
        DRIVE_MOUNTED = True
        print('[✓] Google Drive already mounted.')
    else:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
        print('[✓] Google Drive mounted successfully!')
except Exception as e:
    print(f'[!] Notice: {e}. Falling back to NVMe temporary storage.')

BASE_COMFY = '/content/drive/MyDrive/ComfyUI' if DRIVE_MOUNTED else '/content/ComfyUI'
MODELS_DIR = os.path.join(BASE_COMFY, 'models')
for p in [BASE_COMFY, MODELS_DIR, os.path.join(MODELS_DIR, 'checkpoints')]:
    os.makedirs(p, exist_ok=True)

print(f'[✓] Workspace initialized at: {BASE_COMFY}')


In [ ]:
# @title 3. Install ComfyUI Core Engine & Fast aria2c Tooling
import os, subprocess

!apt-get update -qq && apt-get install -y -qq aria2

# Clone ComfyUI
if not os.path.exists('/content/ComfyUI/main.py'):
    print('[*] Cloning official ComfyUI...')
    !git clone --depth=1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
else:
    print('[✓] ComfyUI already installed.')
    %cd /content/ComfyUI

# Install PyNgrok and Cloudflared
!pip install -q pyngrok websocket-client
if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/usr/bin/cloudflared'):
    !curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
    !dpkg -i cloudflared.deb > /dev/null 2>&1
print('[✓] Core tooling and tunneling packages ready.')


In [ ]:
# @title 3.5. Fast Download Checkpoints via aria2c (Skips Existing Files!)
import os
ckpt_dir = '/content/ComfyUI/models/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

# 1. DreamShaper 8 (1.9 GB - Fast, Ultra-High Quality Image Model)
ds_path = os.path.join(ckpt_dir, 'DreamShaper_8_pruned.safetensors')
if not os.path.exists(ds_path):
    print('[*] Downloading DreamShaper 8...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors' -d {ckpt_dir} -o DreamShaper_8_pruned.safetensors
    print('[✓] DreamShaper 8 ready.')
else:
    print(f'[✓] DreamShaper 8 already exists ({round(os.path.getsize(ds_path)/(1024*1024))} MB) — Skipped.')

# 2. LTX-Video 2.5 (3.8 GB - T4 Optimized Video Model)
ltx_path = os.path.join(ckpt_dir, 'ltx-video-2b-v0.9.1.safetensors')
if not os.path.exists(ltx_path):
    print('[*] Downloading LTX-Video 2.5...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors' -d {ckpt_dir} -o ltx-video-2b-v0.9.1.safetensors
    print('[✓] LTX-Video ready.')
else:
    print(f'[✓] LTX-Video already exists ({round(os.path.getsize(ltx_path)/(1024*1024))} MB) — Skipped.')

# 3. Hunyuan3D-2 Checkpoints (DiT fp16 + Delight + Paint)
hy_path = os.path.join(ckpt_dir, 'hunyuan3d-dit-v2.safetensors')
if not os.path.exists(hy_path):
    print('[*] Downloading Hunyuan3D-2 DiT checkpoint...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-dit-v2-0/model.fp16.safetensors' -d {ckpt_dir} -o hunyuan3d-dit-v2.safetensors
    print('[✓] Hunyuan3D-2 DiT ready.')
else:
    print(f'[✓] Hunyuan3D-2 DiT already exists ({round(os.path.getsize(hy_path)/(1024*1024))} MB) — Skipped.')

delight_path = os.path.join(ckpt_dir, 'hy3d_delight.safetensors')
if not os.path.exists(delight_path):
    print('[*] Downloading Hunyuan3D Delight model...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/kijai/Hunyuan3D-2-ComfyUI/resolve/main/hy3d_delight.safetensors' -d {ckpt_dir} -o hy3d_delight.safetensors
    print('[✓] Delight model ready.')
else:
    print(f'[✓] Delight model already exists ({round(os.path.getsize(delight_path)/(1024*1024))} MB) — Skipped.')

# 4. T5 Text Encoder for LTX-Video (Required for Video Generation prompts)
clip_dir = '/content/ComfyUI/models/clip'
os.makedirs(clip_dir, exist_ok=True)
t5_path = os.path.join(clip_dir, 't5xxl_fp8_e4m3fn.safetensors')
if not os.path.exists(t5_path):
    print('[*] Downloading T5 Text Encoder for LTX-Video...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' -d {clip_dir} -o t5xxl_fp8_e4m3fn.safetensors
    print('[✓] T5 Text Encoder ready.')
else:
    print(f'[✓] T5 Text Encoder already exists ({round(os.path.getsize(t5_path)/(1024*1024))} MB) — Skipped.')

# 5. Realistic Vision v5.1 (Hyper-Realistic Humans & DSLR Portraits)
rv_path = os.path.join(ckpt_dir, 'Realistic_Vision_v5.1.safetensors')
if not os.path.exists(rv_path):
    print('[*] Downloading Realistic Vision v5.1...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/SG161222/Realistic_Vision_v5.1_noVAE/resolve/main/Realistic_Vision_v5.1_fp16-no-vae.safetensors' -d {ckpt_dir} -o Realistic_Vision_v5.1.safetensors
    print('[✓] Realistic Vision v5.1 ready.')
else:
    print(f'[✓] Realistic Vision v5.1 already exists ({round(os.path.getsize(rv_path)/(1024*1024))} MB) — Skipped.')

# 6. Juggernaut XL v9 (1024p Cinematic Masterpiece)
jug_path = os.path.join(ckpt_dir, 'Juggernaut-XL_v9.safetensors')
if not os.path.exists(jug_path):
    print('[*] Downloading Juggernaut XL v9...')
    !aria2c -c -x 16 -s 16 -k 1M 'https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors' -d {ckpt_dir} -o Juggernaut-XL_v9.safetensors
    print('[✓] Juggernaut XL v9 ready.')
else:
    print(f'[✓] Juggernaut XL v9 already exists ({round(os.path.getsize(jug_path)/(1024*1024))} MB) — Skipped.')

print('\n[✓] ALL GENERATIVE MODEL CHECKPOINTS & ENCODERS VERIFIED ON DISK:')
!ls -lh {ckpt_dir}
!ls -lh {clip_dir}


In [ ]:
# @title 4. Custom Nodes & Local C++ Compilation (Ninja + No-Build-Isolation)
import os, subprocess

# Install ninja-build compiler accelerator
!apt-get install -y -qq ninja-build

%cd /content/ComfyUI/custom_nodes

# 1. ComfyUI-Manager
if not os.path.exists('ComfyUI-Manager'):
    !git clone --depth=1 https://github.com/ltdrdata/ComfyUI-Manager.git
print('[✓] ComfyUI-Manager ready.')

# 2. ComfyUI-LTXVideo (Required for Video Generation)
if not os.path.exists('ComfyUI-LTXVideo'):
    print('[*] Cloning ComfyUI-LTXVideo...')
    !git clone --depth=1 https://github.com/kijai/ComfyUI-LTXVideo.git
    !pip install -q -r ComfyUI-LTXVideo/requirements.txt
print('[✓] ComfyUI-LTXVideo ready.')

# 3. ComfyUI-Hunyuan3DWrapper & C++ Extensions
if not os.path.exists('ComfyUI-Hunyuan3DWrapper'):
    print('[*] Cloning ComfyUI-Hunyuan3DWrapper...')
    !git clone --depth=1 https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git
    !pip install -q -r ComfyUI-Hunyuan3DWrapper/requirements.txt
print('[✓] ComfyUI-Hunyuan3DWrapper ready.')

# 4. Compile custom_rasterizer inside its local folder
rasterizer_dir = '/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/hy3dgen/texgen/custom_rasterizer'
if os.path.exists(os.path.join(rasterizer_dir, 'setup.py')):
    print('[*] Compiling custom_rasterizer...')
    %cd {rasterizer_dir}
    !NVCC_FLAGS='-allow-unsupported-compiler' pip install -q --no-build-isolation .
    print('[✓] custom_rasterizer compiled.')

# 5. Compile differentiable_renderer if present
diff_dir = '/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/hy3dgen/texgen/differentiable_renderer'
if os.path.exists(os.path.join(diff_dir, 'setup.py')):
    print('[*] Compiling differentiable_renderer...')
    %cd {diff_dir}
    !NVCC_FLAGS='-allow-unsupported-compiler' pip install -q --no-build-isolation .
    print('[✓] differentiable_renderer compiled.')

# 6. Install nvdiffrast preserving PyTorch CUDA headers
print('[*] Installing nvdiffrast...')
!pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

%cd /content/ComfyUI
print('\n[✓] ALL CUSTOM NODES & CUDA EXTENSIONS SUCCESSFULLY COMPILED!')


In [ ]:
# @title 5. Launch ComfyUI Server & Resilient Tunnel (Re-run Safe!)
import os, time, socket, subprocess, re
from google.colab import userdata
from pyngrok import ngrok

# 1. Clean previous zombie processes so re-runs never hang or block
!pkill -9 -f 'main.py' > /dev/null 2>&1
!pkill -9 -f 'cloudflared' > /dev/null 2>&1
time.sleep(1)

# 2. Launch ComfyUI in background
print('[*] Launching ComfyUI engine on 127.0.0.1:8188...')
%cd /content/ComfyUI
!nohup python main.py --dont-print-server --listen 127.0.0.1 --port 8188 > /content/comfy_server.log 2>&1 &

# 3. Wait for port 8188 readiness (Extended to 120s for custom node imports)
ready = False
print('[*] Waiting for ComfyUI to load custom nodes & bind port 8188 (up to 120s)...')
for sec in range(1, 121):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', 8188)) == 0:
            ready = True
            print(f'\n[✓] ComfyUI bound port 8188 successfully in {sec}s!')
            break
    if sec % 10 == 0:
        print(f'  ... Still loading custom nodes and PyTorch CUDA kernels ({sec}s elapsed)')
    time.sleep(1)

if not ready:
    print('[!] ComfyUI start log (last 35 lines):')
    !tail -n 35 /content/comfy_server.log
    raise TimeoutError('ComfyUI took longer than 120 seconds to bind port 8188.')

print('[✓] ComfyUI is active and listening on port 8188!')

# 4. Tunnel Activation (Ngrok primary -> Cloudflared fallback)
public_url = None
active_provider = None

# Try Ngrok if token exists
try:
    ngrok_token = userdata.get('NGROK_TOKEN') if 'NGROK_TOKEN' in userdata.keys() else None
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        tun = ngrok.connect(8188, 'http')
        public_url = tun.public_url.replace('http://', 'https://')
        active_provider = 'Ngrok (Primary)'
except Exception as e:
    print(f'[!] Ngrok: {e}. Switching to Cloudflared fallback...')

# Fallback to Cloudflared
if not public_url:
    !rm -f /content/cf.log
    !nohup cloudflared tunnel --url http://127.0.0.1:8188 --logfile /content/cf.log > /dev/null 2>&1 &
    for _ in range(40):
        if os.path.exists('/content/cf.log'):
            with open('/content/cf.log', 'r') as f:
                content = f.read()
            for word in content.split():
                if 'trycloudflare.com' in word and word.startswith('https://'):
                    public_url = word.strip().strip('"').strip("'").rstrip(')').rstrip(']')
                    active_provider = 'Cloudflared (Auto-Fallback)'
                    break
            if public_url:
                break
        time.sleep(0.5)

    if not public_url and os.path.exists('/content/cf.log'):
        print('[!] Cloudflared log content:')
        !cat /content/cf.log

print('\n' + '='*65)
print(f'  ⚡ PROJECT AARAMBH GATEWAY ACTIVE ({active_provider})')
print(f'  🔗 PUBLIC URL: {public_url}')
print(f'  👉 Copy this URL into your Local Streamlit Studio (http://localhost:8501)!')
print('='*65 + '\n')
